In [1]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The loop keeps calling the model by checking whether the response contains any `function_call` items.

- If the model returns a function call, your code runs the tool, appends the tool result to `messages`, and sets `has_function_calls = True`.
- Then the `while True` loop runs again and sends the updated history back to the model.
- If the model returns only a normal message and no function calls, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is: **no function calls this turn**.


In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [3]:
from opentelemetry.sdk.trace.export.in_memory_span_exporter import (
    InMemorySpanExporter,
)

memory_exporter = InMemorySpanExporter()

provider.add_span_processor(
    SimpleSpanProcessor(memory_exporter)
)

In [4]:
from openai import OpenAI

from gitsource import GithubRepositoryDataReader
from minsearch import Index

from traced_rag import RAGTraced

from dotenv import load_dotenv
load_dotenv()

COMMIT = "8c1834d"

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()
traced_rag = RAGTraced(index=index, llm_client=client)

In [5]:
memory_exporter.clear()

answer = traced_rag.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

spans = memory_exporter.get_finished_spans()

print("Number of spans:", len(spans))
print("Span names:", [span.name for span in spans])

llm_span = next(span for span in spans if span.name == "llm")

print("Input tokens:", llm_span.attributes["input_tokens"])

duration_ms = (
    llm_span.end_time - llm_span.start_time
) / 1_000_000

print("LLM duration:", duration_ms, "ms")

{
    "name": "search",
    "context": {
        "trace_id": "0x42becb8363f8b3d35239c31ecdbec696",
        "span_id": "0xe307d094c8c8d673",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x017e204d509c1233",
    "start_time": "2026-07-23T01:29:50.539495Z",
    "end_time": "2026-07-23T01:29:50.541104Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "e1694bcc-5130-403c-934d-451952d06d91",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x42becb8363f8b3d35239c31ecdbec696",
        "span_id": "0xec791e378c53e72f",
        "trace_state": "[]"
    },
    "kind": "SpanKind